In [ ]:
import numpy as np
from pathlib import Path
import warnings

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from sklearn.ensemble import GradientBoostingRegressor
warnings.filterwarnings("ignore")

DATA_DIR = Path("..") / "initial_data"

In [ ]:
# ======================================
# STRATEGY 1: GP-UCB (for F1, F7)
# ======================================

def suggest_gp_ucb(X, y, beta=2.0, n_candidates=10000, random_state=0):
    """
    GP-UCB: Fit GP, sample candidates, return max(mu + beta*sigma).
    Higher beta = more exploration.
    """
    d = X.shape[1]
    
    kernel = 1.0 * RBF(length_scale=np.ones(d)) + WhiteKernel(noise_level=1e-3)
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=5, random_state=random_state)
    gp.fit(X, y)
    
    rng = np.random.default_rng(random_state)
    candidates = rng.uniform(0, 1, size=(n_candidates, d))
    
    mu, sigma = gp.predict(candidates, return_std=True)
    ucb = mu + beta * sigma
    
    best_idx = np.argmax(ucb)
    return candidates[best_idx]

In [ ]:
# ======================================
# STRATEGY 2: GradientBoosting (for F2)
# ======================================

def suggest_gradientboost(X, y, resolution=0.05, random_state=0):
    """
    Fit GradientBoosting, predict on grid, return max.
    """
    d = X.shape[1]
    
    model = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=random_state)
    model.fit(X, y)
    
    grids = [np.arange(0, 1 + resolution, resolution) for _ in range(d)]
    mesh = np.meshgrid(*grids, indexing='ij')
    candidates = np.column_stack([m.ravel() for m in mesh])
    
    predictions = model.predict(candidates)
    best_idx = np.argmax(predictions)
    
    return candidates[best_idx]

In [ ]:
# ======================================
# STRATEGY 3: Variance-Guided (PCA-Inspired)
# For F3, F6, F8
# ======================================

def suggest_variance_guided(X, y, top_k=5, perturb_scale=0.05, random_state=0):
    """
    PCA-inspired: Lock low-variance dims, perturb high-variance dims.
    
    - Low spread (< 0.08): dimension is 'solved', lock at best value
    - High spread (> 0.15): dimension is uncertain, perturb it
    - Medium: keep at cluster center
    """
    rng = np.random.default_rng(random_state)
    
    # Find top K
    top_indices = np.argsort(y)[-top_k:]
    top_X = X[top_indices]
    
    # Best point and cluster center
    best_point = X[np.argmax(y)]
    cluster_center = top_X.mean(axis=0)
    
    # Compute spread per dimension
    spreads = top_X.max(axis=0) - top_X.min(axis=0)
    
    # Build suggestion
    suggestion = np.zeros(len(best_point))
    
    for i, spread in enumerate(spreads):
        if spread < 0.08:
            # Low variance: LOCK at best point value
            suggestion[i] = best_point[i]
        elif spread > 0.15:
            # High variance: EXPLORE - perturb around best
            suggestion[i] = best_point[i] + rng.uniform(-perturb_scale, perturb_scale)
        else:
            # Medium: use cluster center
            suggestion[i] = cluster_center[i]
    
    return np.clip(suggestion, 0, 1), spreads

In [ ]:
# ======================================
# STRATEGY 4: Cluster Center (for F4)
# ======================================

def suggest_cluster_center(X, y, top_k=5):
    """
    Simple cluster center: mean of top K results.
    """
    top_indices = np.argsort(y)[-top_k:]
    cluster_center = X[top_indices].mean(axis=0)
    return np.clip(cluster_center, 0, 1)

In [ ]:
# ======================================
# STRATEGY 5: Corner Probe (for F5)
# ======================================

def suggest_corner_probe(X, y, probe_dim=3, probe_value=0.95):
    """
    For corner optima: keep most dims at boundary, probe one interior.
    """
    best_point = X[np.argmax(y)].copy()
    
    # Push all to boundary first
    suggestion = np.where(best_point > 0.5, 1.0, 0.0)
    
    # Probe one dimension
    suggestion[probe_dim] = probe_value
    
    return suggestion

In [ ]:
def parse_multiline_lists(filepath):
    with open(filepath) as f:
        content = f.read()
    rounds = []
    buffer = ""
    bracket_count = 0
    for char in content:
        buffer += char
        if char == '[':
            bracket_count += 1
        elif char == ']':
            bracket_count -= 1
            if bracket_count == 0 and buffer.strip():
                rounds.append(buffer.strip())
                buffer = ""
    return rounds

# Load data - UPDATE FILE NAMES HERE
input_lines = parse_multiline_lists("inputs_m23.txt")
output_lines = parse_multiline_lists("outputs_m23.txt")

inputs_rounds = [eval(line, {"np": np, "array": np.array}) for line in input_lines]
outputs_rounds = [eval(line, {"np": np}) for line in output_lines]
n_rounds = len(inputs_rounds)
print(f"Loaded {n_rounds} rounds")

In [ ]:
# ============================
# WEEK 12 STRATEGY MAP
# ============================
#
# F1: GP-UCB (beta=3.0) - Far behind, aggressive exploration
# F2: GradientBoosting - Just worked in W11, continue
# F3: Variance-guided - Lock d1, explore d2/d3
# F4: Cluster center - Poly2 failed, stay safe
# F5: Corner probe - Test [1,1,1,0.95]
# F6: Variance-guided - Lock d2/d4, explore d1/d5
# F7: GP-UCB (beta=2.5) - Behind classmate, need exploration
# F8: Variance-guided - Lock d1/d4/d5, explore d2/d6/d7

print("=" * 60)
print(f"WEEK {n_rounds + 1} QUERIES")
print("=" * 60)

for i in range(1, 9):
    # Load initial + all rounds
    X = np.load(DATA_DIR / f"function_{i}" / "initial_inputs.npy")
    y = np.load(DATA_DIR / f"function_{i}" / "initial_outputs.npy")
    
    for r in range(n_rounds):
        X = np.vstack([X, np.array(inputs_rounds[r][i-1]).reshape(1, -1)])
        y = np.append(y, float(outputs_rounds[r][i-1]))
    
    best_y = y.max()
    
    # Per-function strategy
    if i == 1:
        # F1: GP-UCB aggressive (beta=3.0)
        # Log transform for tiny values
        y_log = np.log10(np.abs(y) + 1e-300)
        best_x = suggest_gp_ucb(X, y_log, beta=3.0, random_state=42)
        strategy = "GP-UCB (beta=3.0, log-scaled)"
    
    elif i == 2:
        # F2: GradientBoosting - worked in W11
        best_x = suggest_gradientboost(X, y, random_state=42)
        strategy = "GradientBoosting"
    
    elif i == 3:
        # F3: Variance-guided - lock d1, explore d2/d3
        best_x, spreads = suggest_variance_guided(X, y, random_state=42)
        strategy = f"Variance-guided (spreads: {[f'{s:.2f}' for s in spreads]})"
    
    elif i == 4:
        # F4: Cluster center - Poly2 failed
        best_x = suggest_cluster_center(X, y)
        strategy = "Cluster center"
    
    elif i == 5:
        # F5: Corner probe - test [1,1,1,0.95]
        best_x = suggest_corner_probe(X, y, probe_dim=3, probe_value=0.95)
        strategy = "Corner probe (d4=0.95)"
    
    elif i == 6:
        # F6: Variance-guided - lock d2/d4, explore d1/d5
        best_x, spreads = suggest_variance_guided(X, y, random_state=42)
        strategy = f"Variance-guided (spreads: {[f'{s:.2f}' for s in spreads]})"
    
    elif i == 7:
        # F7: GP-UCB moderate (beta=2.5)
        best_x = suggest_gp_ucb(X, y, beta=2.5, random_state=42)
        strategy = "GP-UCB (beta=2.5)"
    
    elif i == 8:
        # F8: Variance-guided - lock d1/d4/d5, explore d2/d6/d7
        best_x, spreads = suggest_variance_guided(X, y, random_state=42)
        strategy = f"Variance-guided (spreads: {[f'{s:.2f}' for s in spreads]})"
    
    # Format and check
    query_str = "-".join(f"{v:.6f}" for v in best_x)
    is_dup = np.any(np.all(np.isclose(X, best_x, atol=1e-4), axis=1))
    
    print(f"\nF{i}: {strategy}")
    print(f"  Best so far: {best_y:.6f}")
    print(f"  Query: {query_str}")
    if is_dup:
        print("  WARNING: DUPLICATE!")